# UAPOML Week 3 Problem Set
## Classification, Regression Trees, Ensemble Methods & Regularisation

**Random seed:** `random_state=42` throughout for reproducibility.

**Dataset Notes (alternatives used where competition datasets required ToS acceptance):**
- Q1: `credit_risk_dataset.csv` (laotse/credit-risk-dataset) — binary credit default, similar schema
- Q2: `heart.csv` (johnsmith88/heart-disease-dataset) — same UCI Heart Disease
- Q3: `AmesHousing.csv` (prevek18/ames-housing-dataset) — 82-feature housing dataset with SalePrice
- Q4: `housing.csv` (camnugent/california-housing-prices) — original dataset ✓
- Q5: `creditcard.csv` (mlg-ulb/creditcardfraud) — fraud detection, Class target
- Q6: `all_stocks_5yr.csv` (camnugent/sandp500) — original dataset ✓
- Q7: `diabetes.csv` (uciml/pima-indians-diabetes-database) — original dataset ✓
- Q8: NIFTY-50 individual CSVs (rohanrao/nifty50-stock-market-data) — original dataset ✓
- Bonus: `tested.csv` (heptapod/titanic) — Titanic with Survived column ✓


In [ ]:
# ============================================================
# Global Imports & Settings
# ============================================================
import warnings
warnings.filterwarnings('ignore')
import os
os.chdir(r'D:\Avtansh\Semester 4\Summer Project Stamatics UAPOML\Ass3')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for server execution
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 100

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, TimeSeriesSplit
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import (
    LogisticRegression, RidgeCV, LassoCV, ElasticNetCV, LinearRegression, Lasso
)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, average_precision_score, mean_squared_error, r2_score
)
from sklearn.inspection import permutation_importance
import xgboost as xgb

SEED = 42
np.random.seed(SEED)
print('All libraries loaded. Working dir:', os.getcwd())

---
## Question 1: Logistic Regression for Credit Default Prediction [Easy]

**Summary:** We build a binary classifier using logistic regression to predict credit default (loan_status). We preprocess with median/mode imputation and standardisation, then evaluate with AUC-ROC, ROC curve, confusion matrix, and threshold analysis. The coefficient vector reveals which features most strongly signal default risk.


In [ ]:
# ============================================================
# Q1 – Credit Default Prediction
# Dataset: credit_risk_dataset.csv (laotse/credit-risk-dataset)
# ============================================================
df_credit = pd.read_csv('credit_risk_dataset.csv')
print('Shape:', df_credit.shape)
print('Columns:', df_credit.columns.tolist())

# Target: loan_status (1=default, 0=no default)
y_credit = df_credit['loan_status'].astype(int)
X_credit = df_credit.drop('loan_status', axis=1)

# (a) Handle missing values via median/mode imputation
for col in X_credit.columns:
    if X_credit[col].dtype == 'object':
        X_credit[col].fillna(X_credit[col].mode()[0], inplace=True)
    else:
        X_credit[col].fillna(X_credit[col].median(), inplace=True)

# Encode categoricals
for col in X_credit.select_dtypes(include='object').columns:
    X_credit[col] = LabelEncoder().fit_transform(X_credit[col].astype(str))

print('Target distribution:')
print(y_credit.value_counts(normalize=True).round(4))

In [ ]:
# (b) Stratified 70/15/15 split
X_tr, X_temp, y_tr, y_temp = train_test_split(
    X_credit, y_credit, test_size=0.30, random_state=SEED, stratify=y_credit)
X_val, X_te, y_val, y_te = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)
print(f'Train: {len(X_tr)}, Val: {len(X_val)}, Test: {len(X_te)}')

# (c) Standardise (fit on train only)
scaler_credit = StandardScaler()
X_tr_sc  = scaler_credit.fit_transform(X_tr)
X_val_sc = scaler_credit.transform(X_val)
X_te_sc  = scaler_credit.transform(X_te)

In [ ]:
# (3) Train Logistic Regression
lr_credit = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=SEED)
lr_credit.fit(X_tr_sc, y_tr)

y_pred_credit = lr_credit.predict(X_te_sc)
y_prob_credit = lr_credit.predict_proba(X_te_sc)[:, 1]

metrics_credit = pd.DataFrame({
    'Metric':  ['Accuracy','Precision','Recall','F1-Score','AUC-ROC'],
    'Score': [
        round(accuracy_score(y_te, y_pred_credit), 4),
        round(precision_score(y_te, y_pred_credit), 4),
        round(recall_score(y_te, y_pred_credit), 4),
        round(f1_score(y_te, y_pred_credit), 4),
        round(roc_auc_score(y_te, y_prob_credit), 4)
    ]
})
print('\n--- Test Metrics: Logistic Regression (Credit Default) ---')
print(metrics_credit.to_string(index=False))

In [ ]:
# (4) ROC Curve + Confusion Matrix
fpr, tpr, _ = roc_curve(y_te, y_prob_credit)
auc_val = roc_auc_score(y_te, y_prob_credit)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc_val:.4f}')
axes[0].plot([0,1],[0,1],'k--', lw=1)
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve – Logistic Regression (Credit Default)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

cm = confusion_matrix(y_te, y_pred_credit)
ConfusionMatrixDisplay(cm, display_labels=['No Default','Default']).plot(ax=axes[1], colorbar=False)
axes[1].set_title('Confusion Matrix – Logistic Regression')
plt.tight_layout()
plt.savefig('q1_roc_cm.png', bbox_inches='tight'); plt.show()
print('Saved: q1_roc_cm.png')

In [ ]:
# (5) Threshold analysis: Precision, Recall, F1 vs tau
thresholds = np.arange(0.1, 1.0, 0.1)
precisions, recalls, f1s = [], [], []
for t in thresholds:
    preds_t = (y_prob_credit >= t).astype(int)
    precisions.append(precision_score(y_te, preds_t, zero_division=0))
    recalls.append(recall_score(y_te, preds_t, zero_division=0))
    f1s.append(f1_score(y_te, preds_t, zero_division=0))

best_idx = np.argmax(f1s)
plt.figure(figsize=(9, 5))
plt.plot(thresholds, precisions, 'o-', color='royalblue', label='Precision')
plt.plot(thresholds, recalls,    's-', color='tomato',    label='Recall')
plt.plot(thresholds, f1s,        '^-', color='green',     label='F1-Score')
plt.axvline(thresholds[best_idx], color='purple', linestyle='--',
            label=f'Best F1 threshold = {thresholds[best_idx]:.1f}')
plt.xlabel('Classification Threshold (τ)'); plt.ylabel('Score')
plt.title('Precision, Recall, F1 vs. Threshold – Credit Default')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('q1_threshold.png', bbox_inches='tight'); plt.show()
print(f'Best F1 = {f1s[best_idx]:.4f} at τ = {thresholds[best_idx]:.1f}')

In [ ]:
# (6) Top-5 features by |coefficient|
coef = lr_credit.coef_[0]
coef_df = pd.DataFrame({'Feature': X_credit.columns, 'Coefficient': coef})
top5 = coef_df.reindex(coef_df['Coefficient'].abs().nlargest(5).index)[['Feature','Coefficient']]
print('\n--- Top 5 Features by |Coefficient| ---')
print(top5.to_string(index=False))

plt.figure(figsize=(8,4))
colors = ['tomato' if c < 0 else 'steelblue' for c in top5['Coefficient']]
plt.barh(top5['Feature'], top5['Coefficient'], color=colors)
plt.xlabel('Coefficient Value')
plt.title('Top 5 Logistic Regression Coefficients – Credit Default')
plt.tight_layout()
plt.savefig('q1_coef.png', bbox_inches='tight'); plt.show()

### Q1 – Interpretation of Coefficient Magnitudes

The five features with the largest absolute coefficients reveal the dominant predictors of credit default. **`loan_percent_income`** (loan amount as a fraction of income) typically carries the largest positive coefficient — borrowers whose loan obligations consume a large fraction of income are at acute risk if income falls. **`loan_int_rate`** (interest rate) is highly predictive because lenders price risk into the rate; a high rate both signals prior risk assessment and directly increases repayment burden. **`loan_grade`** encodes the lender's own creditworthiness assessment, making it a strong predictor by construction. **`cb_person_default_on_file`** (prior default on record) has a strong positive coefficient — historical default is the single most predictive feature of future default, consistent with credit-scoring literature. **`person_income`** typically has a negative coefficient — higher income reduces default probability by providing a larger buffer against financial shocks. These findings align with standard credit-scoring models such as FICO and validate our logistic regression's financial interpretability.


---
## Question 2: Decision Tree Classifier with Pruning for Heart Disease [Medium]

**Summary:** We apply decision tree classification to the Heart Disease UCI dataset. Decision trees are highly interpretable — clinicians can trace which physiological features (chest pain type, max heart rate, ST depression) drive predictions. We compare unpruned vs. depth-limited pre-pruning vs. cost-complexity post-pruning to study the bias–variance trade-off.


In [ ]:
# ============================================================
# Q2 – Heart Disease (johnsmith88/heart-disease-dataset)
# ============================================================
df_heart = pd.read_csv('heart.csv')
print('Shape:', df_heart.shape)
print('Columns:', df_heart.columns.tolist())

X_heart = df_heart.drop('target', axis=1)
y_heart = df_heart['target']

# Stratified 70/15/15 split
X_htr, X_htemp, y_htr, y_htemp = train_test_split(
    X_heart, y_heart, test_size=0.30, random_state=SEED, stratify=y_heart)
X_hval, X_hte, y_hval, y_hte = train_test_split(
    X_htemp, y_htemp, test_size=0.50, random_state=SEED, stratify=y_htemp)
print(f'Train: {len(X_htr)}, Val: {len(X_hval)}, Test: {len(X_hte)}')

In [ ]:
# (3) Unpruned tree
dt_unpruned = DecisionTreeClassifier(max_depth=None, random_state=SEED)
dt_unpruned.fit(X_htr, y_htr)
print(f'Unpruned – Train Acc: {accuracy_score(y_htr, dt_unpruned.predict(X_htr)):.4f}')
print(f'Unpruned – Test  Acc: {accuracy_score(y_hte, dt_unpruned.predict(X_hte)):.4f}')
print('=> Gap between train and test accuracy indicates overfitting.')

In [ ]:
# (4) Depth sweep
depths = [1, 2, 3, 5, 7, 10, 15, None]
train_accs, val_accs = [], []
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=SEED)
    dt.fit(X_htr, y_htr)
    train_accs.append(accuracy_score(y_htr, dt.predict(X_htr)))
    val_accs.append(accuracy_score(y_hval, dt.predict(X_hval)))

depth_labels = [str(d) if d is not None else 'None' for d in depths]
best_depth_idx = np.argmax(val_accs)
best_depth = depths[best_depth_idx]
print(f'Best depth (val): {best_depth}  (val acc = {val_accs[best_depth_idx]:.4f})')

plt.figure(figsize=(9, 5))
plt.plot(depth_labels, train_accs, 'o-', color='royalblue', label='Train Accuracy')
plt.plot(depth_labels, val_accs,   's-', color='tomato',    label='Val Accuracy')
plt.axvline(depth_labels[best_depth_idx], color='green', linestyle='--',
            label=f'Best depth = {best_depth}')
plt.xlabel('max_depth'); plt.ylabel('Accuracy')
plt.title('Decision Tree Depth vs. Accuracy – Heart Disease')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('q2_depth_curve.png', bbox_inches='tight'); plt.show()

In [ ]:
# (5) Best depth-limited tree retrained on train+val
X_htrval = pd.concat([X_htr, X_hval])
y_htrval = pd.concat([y_htr, y_hval])
dt_depth = DecisionTreeClassifier(max_depth=best_depth, random_state=SEED)
dt_depth.fit(X_htrval, y_htrval)
auc_depth = roc_auc_score(y_hte, dt_depth.predict_proba(X_hte)[:,1])
print(f'Depth-limited (max_depth={best_depth}) Test AUC-ROC: {auc_depth:.4f}')

In [ ]:
# (6) Cost-Complexity Pruning
dt_full = DecisionTreeClassifier(random_state=SEED)
path = dt_full.cost_complexity_pruning_path(X_htr, y_htr)
ccp_alphas = path.ccp_alphas

val_aucs_ccp = []
for alpha in ccp_alphas:
    dt_c = DecisionTreeClassifier(ccp_alpha=alpha, random_state=SEED)
    dt_c.fit(X_htr, y_htr)
    prob = dt_c.predict_proba(X_hval)
    if prob.shape[1] == 2:
        val_aucs_ccp.append(roc_auc_score(y_hval, prob[:,1]))
    else:
        val_aucs_ccp.append(0.5)

best_alpha_idx = np.argmax(val_aucs_ccp)
best_alpha = ccp_alphas[best_alpha_idx]
print(f'Best ccp_alpha: {best_alpha:.6f}  (val AUC = {val_aucs_ccp[best_alpha_idx]:.4f})')

dt_pruned = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=SEED)
dt_pruned.fit(X_htrval, y_htrval)
auc_pruned = roc_auc_score(y_hte, dt_pruned.predict_proba(X_hte)[:,1])
print(f'Cost-Complexity Pruned Test AUC-ROC: {auc_pruned:.4f}')

In [ ]:
# (7) Visualise pruned tree (max 4 levels)
plt.figure(figsize=(20, 8))
plot_tree(dt_pruned, feature_names=X_heart.columns.tolist(),
          class_names=['No Disease','Disease'], filled=True, max_depth=4, fontsize=9)
plt.title('Pruned Decision Tree (Cost-Complexity) – Heart Disease (max 4 levels)')
plt.tight_layout()
plt.savefig('q2_tree.png', bbox_inches='tight'); plt.show()

In [ ]:
# Comparison table
auc_unpruned = roc_auc_score(y_hte, dt_unpruned.predict_proba(X_hte)[:,1])
comp_q2 = pd.DataFrame({
    'Model':      ['Unpruned', f'Depth-limited (d={best_depth})', 'Cost-Complexity Pruned'],
    'AUC-ROC':    [round(auc_unpruned,4), round(auc_depth,4), round(auc_pruned,4)],
    'F1-Score':   [round(f1_score(y_hte, dt_unpruned.predict(X_hte)),4),
                   round(f1_score(y_hte, dt_depth.predict(X_hte)),4),
                   round(f1_score(y_hte, dt_pruned.predict(X_hte)),4)],
    'Tree Depth': [dt_unpruned.get_depth(), dt_depth.get_depth(), dt_pruned.get_depth()]
})
print('\n--- Q2 Comparison Table ---')
print(comp_q2.to_string(index=False))
winner = 'Cost-Complexity Pruned' if auc_pruned >= auc_depth else f'Depth-limited (d={best_depth})'
print(f'\nWinner: {winner}')
print('Cost-complexity pruning avoids the horizon effect by growing the full tree first, '
      'then globally optimising the complexity–purity trade-off.')

---
## Question 3: Regularised Regression for House Price Prediction [Medium]

**Summary:** We apply Ridge, Lasso, and Elastic Net regularisation to the Ames Housing dataset (82 features, SalePrice target). Multicollinearity among structural features makes this an ideal testbed. We use 5-fold CV to tune regularisation strength, compare RMSE and R², and visualise the Lasso coefficient path to study feature selection.


In [ ]:
# ============================================================
# Q3 – Ames Housing (prevek18/ames-housing-dataset)
# ============================================================
df_house = pd.read_csv('AmesHousing.csv')
print('Shape:', df_house.shape)

# Target: SalePrice (log-transform)
y_house = np.log1p(df_house['SalePrice'])
X_house = df_house.drop('SalePrice', axis=1)
# Drop Order/PID if present
X_house = X_house.drop([c for c in X_house.columns if c in ['Order','PID']], axis=1, errors='ignore')

# (b) Drop features with >40% missing
X_house = X_house.loc[:, X_house.isnull().mean() <= 0.40]
print(f'After dropping high-missing: {X_house.shape[1]} features')

# (c) Impute: median for numeric, mode for categorical
for col in X_house.columns:
    if X_house[col].dtype == 'object':
        X_house[col].fillna(X_house[col].mode()[0], inplace=True)
    else:
        X_house[col].fillna(X_house[col].median(), inplace=True)

# (d) One-hot encode
X_house = pd.get_dummies(X_house)
print(f'After one-hot encoding: {X_house.shape[1]} features')

# 80/20 split (CV does tuning)
X_htr2, X_hte2, y_htr2, y_hte2 = train_test_split(
    X_house, y_house, test_size=0.20, random_state=SEED)

# (e) Standardise (fit on train only)
scaler_house = StandardScaler()
X_htr2_sc = scaler_house.fit_transform(X_htr2)
X_hte2_sc = scaler_house.transform(X_hte2)
print(f'Train: {X_htr2_sc.shape}, Test: {X_hte2_sc.shape}')

In [ ]:
# (3) Train Ridge, Lasso, Elastic Net with 5-fold CV
alphas_grid = np.logspace(-3, 3, 50)

ridge_cv = RidgeCV(alphas=alphas_grid, cv=5)
ridge_cv.fit(X_htr2_sc, y_htr2)

lasso_cv = LassoCV(alphas=alphas_grid, cv=5, random_state=SEED, max_iter=10000)
lasso_cv.fit(X_htr2_sc, y_htr2)

enet_cv = ElasticNetCV(l1_ratio=[0.1,0.5,0.7,0.9,0.95,1.0], alphas=alphas_grid,
                       cv=5, random_state=SEED, max_iter=10000)
enet_cv.fit(X_htr2_sc, y_htr2)

ols = LinearRegression()
ols.fit(X_htr2_sc, y_htr2)

def eval_reg(model, Xte, yte, name):
    preds = model.predict(Xte)
    rmse_log = np.sqrt(mean_squared_error(yte, preds))
    r2 = r2_score(yte, preds)
    rmse_orig = np.sqrt(mean_squared_error(np.expm1(yte), np.expm1(preds)))
    nz = int(np.sum(model.coef_ != 0)) if hasattr(model,'coef_') else 'N/A'
    alph = getattr(model, 'alpha_', 'N/A')
    return {'Model': name, 'Best α': round(float(alph),5) if alph != 'N/A' else 'N/A',
            'Non-zero Coef': nz, 'RMSE (log)': round(rmse_log,5),
            'RMSE ($)': f'{rmse_orig:,.0f}', 'R²': round(r2,5)}

results_house = pd.DataFrame([
    eval_reg(ols,      X_hte2_sc, y_hte2, 'OLS'),
    eval_reg(ridge_cv, X_hte2_sc, y_hte2, 'Ridge'),
    eval_reg(lasso_cv, X_hte2_sc, y_hte2, 'Lasso'),
    eval_reg(enet_cv,  X_hte2_sc, y_hte2, 'Elastic Net'),
])
print('\n--- Q3: Regularised Regression Comparison Table ---')
print(results_house.to_string(index=False))

In [ ]:
# (5) Lasso Coefficient Path
alphas_path = np.logspace(-4, 1, 80)
coef_paths  = []
for a in alphas_path:
    lp = Lasso(alpha=a, max_iter=10000, random_state=SEED)
    lp.fit(X_htr2_sc, y_htr2)
    coef_paths.append(lp.coef_.copy())
coef_paths = np.array(coef_paths)

feature_names_arr = np.array(X_house.columns.tolist())

# Find last 5 features to reach zero
last_nonzero_alpha_idx = []
for fi in range(coef_paths.shape[1]):
    nz = np.where(coef_paths[:, fi] != 0)[0]
    if len(nz) > 0:
        last_nonzero_alpha_idx.append((nz[-1], fi))
last_five = sorted(last_nonzero_alpha_idx, reverse=True)[:5]

plt.figure(figsize=(12, 6))
for fi in range(coef_paths.shape[1]):
    plt.plot(np.log(alphas_path), coef_paths[:, fi], color='lightblue', alpha=0.35, lw=0.7)

annotated = set()
colors_ann = ['red','green','orange','purple','brown']
for i, (_, fi) in enumerate(last_five):
    nz = np.where(coef_paths[:, fi] != 0)[0]
    if len(nz) > 0 and fi not in annotated:
        x_ann = np.log(alphas_path[nz[-1]])
        y_ann = coef_paths[nz[-1], fi]
        fname = feature_names_arr[fi] if fi < len(feature_names_arr) else f'F{fi}'
        plt.plot(np.log(alphas_path), coef_paths[:, fi], lw=1.5, color=colors_ann[i])
        plt.annotate(fname[:20], (x_ann, y_ann), fontsize=7,
                     xytext=(6,4), textcoords='offset points', color=colors_ann[i])
        annotated.add(fi)

plt.axvline(np.log(lasso_cv.alpha_), color='red', linestyle='--',
            label=f'Best α = {lasso_cv.alpha_:.5f}')
plt.xlabel('log(α)'); plt.ylabel('Coefficient Value')
plt.title('Lasso Coefficient Path – House Prices')
plt.legend(); plt.tight_layout()
plt.savefig('q3_lasso_path.png', bbox_inches='tight'); plt.show()

### Q3 – Why Lasso Produces Sparse Solutions (Geometric Explanation)

The L1 constraint region (Lasso) is a **hyperdiamond** whose corners lie exactly on the coordinate axes. The squared-error loss forms elliptical contours in coefficient space. The regularised solution is the point where these contours first touch the constraint region. Because the L1 hyperdiamond has corners on the axes, the tangent point is geometrically far more likely to occur at a corner — where one or more coefficients are exactly zero. The smooth L2 sphere (Ridge) has no corners; its tangent point almost never falls on a coordinate axis, so Ridge rarely zeroes coefficients. In noisy financial datasets, Lasso's sparsity is valuable: it drives irrelevant features to zero, performing automatic feature selection and reducing overfitting to noise.


---
## Question 4: Locally Weighted Regression on Non-Linear Data [Medium]

**Summary:** We implement LWR entirely from scratch using NumPy on California Housing (median_income → median_house_value). LWR fits a local linear model at each query point via a Gaussian kernel. We sweep five bandwidths to study bias–variance trade-off and compare against OLS on the same 1D problem.


In [ ]:
# ============================================================
# Q4 – LWR from scratch (NumPy ONLY)
# Dataset: housing.csv (California Housing)
# ============================================================
df_cal = pd.read_csv('housing.csv')
df_cal.dropna(inplace=True)
print('Shape:', df_cal.shape)

# Sample 2000 rows
rng = np.random.RandomState(SEED)
sample_idx = rng.choice(len(df_cal), 2000, replace=False)
df_cal_s = df_cal.iloc[sample_idx]

X_cal_1d = df_cal_s['median_income'].values
y_cal    = df_cal_s['median_house_value'].values

# 70/30 split
n_total = len(X_cal_1d)
n_train = int(0.70 * n_total)
idx_perm = rng.permutation(n_total)
X_cal_tr = X_cal_1d[idx_perm[:n_train]]
y_cal_tr = y_cal[idx_perm[:n_train]]
X_cal_te = X_cal_1d[idx_perm[n_train:]]
y_cal_te = y_cal[idx_perm[n_train:]]
print(f'Train: {len(X_cal_tr)}, Test: {len(X_cal_te)}')

In [ ]:
# LWR implementation (NumPy only)
def lwr_predict_point(x_q, X_tr, y_tr, tau):
    """Locally Weighted Regression at a single query point."""
    w = np.exp(-((X_tr - x_q) ** 2) / (2 * tau ** 2))
    W = np.diag(w)
    X_b = np.column_stack([np.ones(len(X_tr)), X_tr])
    XtW = X_b.T @ W
    try:
        theta = np.linalg.solve(XtW @ X_b, XtW @ y_tr)
    except np.linalg.LinAlgError:
        theta = np.linalg.lstsq(XtW @ X_b, XtW @ y_tr, rcond=None)[0]
    return np.array([1.0, x_q]) @ theta

def lwr_predict(X_query, X_tr, y_tr, tau):
    return np.array([lwr_predict_point(xq, X_tr, y_tr, tau) for xq in X_query])

print('LWR (NumPy-only) functions defined.')

In [ ]:
# Bandwidth sweep + OLS comparison
taus = [0.05, 0.1, 0.3, 1.0, 3.0]
X_plot = np.linspace(X_cal_tr.min(), X_cal_tr.max(), 100)  # 100 pts for speed
rmse_lwr = {}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, tau in enumerate(taus):
    y_plot_pred = lwr_predict(X_plot, X_cal_tr, y_cal_tr, tau)
    y_te_pred   = lwr_predict(X_cal_te, X_cal_tr, y_cal_tr, tau)
    rmse = np.sqrt(np.mean((y_cal_te - y_te_pred) ** 2))
    rmse_lwr[tau] = rmse

    axes[i].scatter(X_cal_tr, y_cal_tr, s=5, alpha=0.3, color='lightblue')
    axes[i].plot(X_plot, y_plot_pred, color='tomato', lw=2)
    axes[i].set_xlabel('Median Income'); axes[i].set_ylabel('Median House Value ($)')
    axes[i].set_title(f'LWR τ={tau} | RMSE={rmse:,.0f}')
    axes[i].grid(True, alpha=0.3)

# OLS in last subplot
X_ols_tr = np.column_stack([np.ones(len(X_cal_tr)), X_cal_tr])
X_ols_te = np.column_stack([np.ones(len(X_cal_te)), X_cal_te])
theta_ols = np.linalg.lstsq(X_ols_tr, y_cal_tr, rcond=None)[0]
rmse_ols  = np.sqrt(np.mean((y_cal_te - X_ols_te @ theta_ols) ** 2))

y_ols_plot = np.column_stack([np.ones(len(X_plot)), X_plot]) @ theta_ols
axes[5].scatter(X_cal_tr, y_cal_tr, s=5, alpha=0.3, color='lightblue')
axes[5].plot(X_plot, y_ols_plot, color='purple', lw=2)
axes[5].set_xlabel('Median Income'); axes[5].set_ylabel('Median House Value ($)')
axes[5].set_title(f'OLS | RMSE={rmse_ols:,.0f}')
axes[5].grid(True, alpha=0.3)

plt.suptitle('LWR vs OLS – California Housing (1D: Median Income)', fontsize=13)
plt.tight_layout()
plt.savefig('q4_lwr.png', bbox_inches='tight'); plt.show()

print('\n--- Q4: RMSE Comparison ---')
for t, r in rmse_lwr.items():
    print(f'  LWR τ={t}: RMSE = {r:,.0f}')
print(f'  OLS:      RMSE = {rmse_ols:,.0f}')
best_tau = min(rmse_lwr, key=rmse_lwr.get)
print(f'\nBest bandwidth: τ = {best_tau}')

### Q4 – Analysis

**Role of τ:** Small τ → very local fit (low bias, high variance — overfitting to noise). Large τ → near-global fit (high bias, low variance — underfitting). **Mathematical limit as τ → ∞:** All kernel weights wᵢ = exp(−||xᵢ−x_q||²/2τ²) → 1, so W → I, and weighted least squares θ̂=(X'WX)⁻¹X'Wy reduces to OLS θ̂=(X'X)⁻¹X'y. LWR with τ→∞ is exactly OLS.

**Practical difficulties in higher dimensions:** The curse of dimensionality shrinks the effective neighbourhood exponentially with d, requiring exponentially more data. Computation scales as O(n²d) per query point, making LWR infeasible for large high-dimensional datasets.


---
## Question 5: Random Forest with Feature Importance for Fraud Detection [Hard]

**Summary:** We apply Random Forest to the Credit Card Fraud dataset (PCA-anonymised features, Class target). We use class_weight='balanced' to handle severe class imbalance (~0.17% fraud). We compare MDI vs. Permutation Importance and justify why AUC-PR is more informative than AUC-ROC under imbalance.


In [ ]:
# ============================================================
# Q5 – Credit Card Fraud (mlg-ulb/creditcardfraud)
# Features: V1-V28 (PCA), Time, Amount; Target: Class
# ============================================================
df_fraud = pd.read_csv('creditcard.csv')
print('Full shape:', df_fraud.shape)
print('Fraud rate:', df_fraud['Class'].mean().round(5))

# Sample 50,000 rows
df_fraud = df_fraud.sample(50000, random_state=SEED).reset_index(drop=True)
print('Sampled shape:', df_fraud.shape)

X_fraud = df_fraud.drop('Class', axis=1)
y_fraud = df_fraud['Class']

# Pre-processing: no missing values in creditcard.csv; just split
X_ftr, X_ftemp, y_ftr, y_ftemp = train_test_split(
    X_fraud, y_fraud, test_size=0.30, random_state=SEED, stratify=y_fraud)
X_fval, X_fte, y_fval, y_fte = train_test_split(
    X_ftemp, y_ftemp, test_size=0.50, random_state=SEED, stratify=y_ftemp)
print(f'Train: {len(X_ftr)}, Val: {len(X_fval)}, Test: {len(X_fte)}')
print(f'Fraud rate in test: {y_fte.mean():.5f}')

In [ ]:
# (3) RandomForest with OOB score
rf_base = RandomForestClassifier(
    n_estimators=200, oob_score=True, class_weight='balanced',
    random_state=SEED, n_jobs=-1)
rf_base.fit(X_ftr, y_ftr)
print(f'OOB Score (initial RF, 200 trees): {rf_base.oob_score_:.4f}')

In [ ]:
# (4) Tune max_features via 5-fold stratified CV
max_features_options = ['sqrt', 'log2', 0.3, 0.5]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_aucs = {}
for mf in max_features_options:
    rf_tmp = RandomForestClassifier(
        n_estimators=100, max_features=mf, class_weight='balanced',
        random_state=SEED, n_jobs=-1)
    scores = cross_val_score(rf_tmp, X_ftr, y_ftr, cv=skf, scoring='roc_auc', n_jobs=-1)
    cv_aucs[str(mf)] = round(scores.mean(), 4)
    print(f'  max_features={str(mf):6} → mean AUC-ROC = {scores.mean():.4f}')

best_mf_str = max(cv_aucs, key=cv_aucs.get)
best_mf = float(best_mf_str) if best_mf_str not in ['sqrt','log2'] else best_mf_str
print(f'Best max_features: {best_mf}')

rf_best = RandomForestClassifier(
    n_estimators=200, max_features=best_mf, oob_score=True,
    class_weight='balanced', random_state=SEED, n_jobs=-1)
rf_best.fit(X_ftr, y_ftr)
print(f'OOB Score (best RF): {rf_best.oob_score_:.4f}')

In [ ]:
# (5) MDI vs Permutation Importance
mdi_df = pd.DataFrame({'Feature': X_fraud.columns,
                        'MDI': rf_best.feature_importances_}).nlargest(20,'MDI')

perm_result = permutation_importance(
    rf_best, X_fval, y_fval, n_repeats=10,
    scoring='roc_auc', random_state=SEED, n_jobs=-1)
perm_df = pd.DataFrame({'Feature': X_fraud.columns,
                         'Perm': perm_result.importances_mean}).nlargest(20,'Perm')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].barh(mdi_df['Feature'][::-1], mdi_df['MDI'][::-1], color='steelblue')
axes[0].set_xlabel('Mean Decrease in Impurity')
axes[0].set_title('Top-20 Features – MDI (Fraud Detection)')
axes[0].grid(True, alpha=0.3)

axes[1].barh(perm_df['Feature'][::-1], perm_df['Perm'][::-1], color='tomato')
axes[1].set_xlabel('Mean AUC Decrease (Permutation)')
axes[1].set_title('Top-20 Features – Permutation Importance (Fraud Detection)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('q5_feature_importance.png', bbox_inches='tight'); plt.show()
print('MDI top feature:', mdi_df.iloc[0]['Feature'])
print('Perm top feature:', perm_df.iloc[0]['Feature'])
print('Agreement:', 'YES' if mdi_df.iloc[0]['Feature']==perm_df.iloc[0]['Feature'] else 'PARTIAL - likely MDI bias toward high-cardinality features')

In [ ]:
# (6) Test set evaluation
y_fte_prob = rf_best.predict_proba(X_fte)[:, 1]
auc_roc_fraud = roc_auc_score(y_fte, y_fte_prob)
ap_fraud      = average_precision_score(y_fte, y_fte_prob)
print(f'Test AUC-ROC:           {auc_roc_fraud:.4f}')
print(f'Test AUC-PR (Avg Prec): {ap_fraud:.4f}')
print('\nAUC-PR is more informative under class imbalance: AUC-ROC is inflated by the '
      'massive true-negative count, while AUC-PR focuses on precision and recall of the '
      'positive (fraud) class we actually care about.')

prec_c, rec_c, _ = precision_recall_curve(y_fte, y_fte_prob)
plt.figure(figsize=(8, 5))
plt.plot(rec_c, prec_c, color='steelblue', lw=2, label=f'AP = {ap_fraud:.4f}')
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curve – Fraud Detection (Random Forest)')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('q5_pr_curve.png', bbox_inches='tight'); plt.show()

In [ ]:
# (7) OOB error as function of n_estimators
n_est_range = list(range(1, 201, 5))
oob_errors = []
for n in n_est_range:
    rf_oob = RandomForestClassifier(
        n_estimators=n, max_features=best_mf, oob_score=True,
        class_weight='balanced', random_state=SEED, n_jobs=-1)
    rf_oob.fit(X_ftr, y_ftr)
    oob_errors.append(1 - rf_oob.oob_score_)

plateau_idx = int(np.argmin(np.gradient(np.gradient(oob_errors))))
plt.figure(figsize=(9, 5))
plt.plot(n_est_range, oob_errors, color='steelblue', lw=2)
plt.axvline(n_est_range[min(plateau_idx, len(n_est_range)-1)],
            color='red', linestyle='--',
            label=f'Approx plateau @ {n_est_range[min(plateau_idx, len(n_est_range)-1)]} trees')
plt.xlabel('n_estimators'); plt.ylabel('OOB Error Rate')
plt.title('OOB Error vs. n_estimators – Fraud Detection')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('q5_oob.png', bbox_inches='tight'); plt.show()
print(f'OOB error plateaus around n_estimators ≈ {n_est_range[min(plateau_idx, len(n_est_range)-1)]}')

---
## Question 6: XGBoost for Stock Return Classification [Hard]

**Summary:** We use XGBoost to predict whether S&P 500 stocks will produce a positive 21-day forward return. We enforce a strict time-based split (train: pre-2017, test: 2017+) to avoid look-ahead bias, perform manual grid search, and analyse feature importances by gain.


In [ ]:
# ============================================================
# Q6 – XGBoost (S&P 500, camnugent/sandp500)
# ============================================================
df_sp500 = pd.read_csv('all_stocks_5yr.csv', parse_dates=['date'])
print('Shape:', df_sp500.shape)

df_sp500 = df_sp500.sort_values(['Name','date']).reset_index(drop=True)
df_sp500['daily_ret'] = df_sp500.groupby('Name')['close'].pct_change()

# Feature engineering
df_sp500['ret_5d']  = df_sp500.groupby('Name')['close'].pct_change(5)
df_sp500['ret_10d'] = df_sp500.groupby('Name')['close'].pct_change(10)
df_sp500['ret_21d'] = df_sp500.groupby('Name')['close'].pct_change(21)
df_sp500['vol_21d'] = df_sp500.groupby('Name')['daily_ret'].transform(
    lambda x: x.rolling(21).std())
df_sp500['vol_price'] = df_sp500['volume'] / (df_sp500['close'] + 1e-8)

def rsi(series, w=14):
    d = series.diff()
    gain = d.clip(lower=0).rolling(w).mean()
    loss = (-d.clip(upper=0)).rolling(w).mean()
    return 100 - 100/(1 + gain/(loss+1e-8))

df_sp500['rsi_14'] = df_sp500.groupby('Name')['close'].transform(rsi)
df_sp500['fwd_21d'] = df_sp500.groupby('Name')['close'].transform(
    lambda x: x.pct_change(21).shift(-21))
df_sp500['target'] = (df_sp500['fwd_21d'] > 0).astype(int)

feat_cols = ['ret_5d','ret_10d','ret_21d','vol_21d','vol_price','rsi_14']
df_sp500.dropna(subset=feat_cols+['target'], inplace=True)
print(f'Clean: {df_sp500.shape}')

In [ ]:
# Time-based split
cutoff = pd.Timestamp('2017-01-01')
tr_m = df_sp500['date'] < cutoff
te_m = df_sp500['date'] >= cutoff

X_sp_tr = df_sp500.loc[tr_m, feat_cols]
y_sp_tr = df_sp500.loc[tr_m, 'target']
X_sp_te = df_sp500.loc[te_m, feat_cols]
y_sp_te = df_sp500.loc[te_m, 'target']

print(f'Train: {len(X_sp_tr)} | Test: {len(X_sp_te)}')
print(f'Train period: {df_sp500.loc[tr_m,"date"].min().date()} → {df_sp500.loc[tr_m,"date"].max().date()}')
print(f'Test  period: {df_sp500.loc[te_m,"date"].min().date()} → {df_sp500.loc[te_m,"date"].max().date()}')

# Early stopping eval set: last 6 months of training
es_cutoff = df_sp500.loc[tr_m,'date'].max() - pd.DateOffset(months=6)
es_mask   = df_sp500.loc[tr_m,'date'] >= es_cutoff
X_es = X_sp_tr[es_mask]; y_es = y_sp_tr[es_mask]
X_tr2 = X_sp_tr[~es_mask]; y_tr2 = y_sp_tr[~es_mask]

In [ ]:
# Manual grid search
param_grid_sp = {
    'max_depth':     [3, 5],
    'learning_rate': [0.05, 0.1],
    'subsample':     [0.6, 0.8]
}
grid_res = []
for md in param_grid_sp['max_depth']:
    for lr in param_grid_sp['learning_rate']:
        for ss in param_grid_sp['subsample']:
            m = xgb.XGBClassifier(n_estimators=200, max_depth=md, learning_rate=lr,
                                   subsample=ss, eval_metric='auc', early_stopping_rounds=20,
                                   random_state=SEED, n_jobs=-1, verbosity=0)
            m.fit(X_tr2, y_tr2, eval_set=[(X_es, y_es)], verbose=False)
            val_auc = roc_auc_score(y_es, m.predict_proba(X_es)[:,1])
            grid_res.append({'max_depth':md,'learning_rate':lr,'subsample':ss,
                             'val_AUC':round(val_auc,4)})

grid_df = pd.DataFrame(grid_res).sort_values('val_AUC', ascending=False)
print('\n--- Grid Search Results (Top 10) ---')
print(grid_df.head(10).to_string(index=False))

In [ ]:
# Best model on test set
bp = grid_df.iloc[0]
xgb_final = xgb.XGBClassifier(
    n_estimators=300, max_depth=int(bp['max_depth']),
    learning_rate=bp['learning_rate'], subsample=bp['subsample'],
    eval_metric='auc', early_stopping_rounds=30,
    random_state=SEED, n_jobs=-1, verbosity=0)
xgb_final.fit(X_tr2, y_tr2, eval_set=[(X_es, y_es)], verbose=False)

y_sp_prob = xgb_final.predict_proba(X_sp_te)[:,1]
auc_sp = roc_auc_score(y_sp_te, y_sp_prob)
ap_sp  = average_precision_score(y_sp_te, y_sp_prob)
print(f'Best params: depth={int(bp["max_depth"])}, lr={bp["learning_rate"]}, ss={bp["subsample"]}')
print(f'Test AUC-ROC: {auc_sp:.4f}  |  Test AUC-PR: {ap_sp:.4f}')
print(f'Naive majority baseline AUC-ROC: 0.5000  |  AUC-PR: {y_sp_te.mean():.4f}')

In [ ]:
# Feature importance by gain
fi_gain = xgb_final.get_booster().get_score(importance_type='gain')
fi_df = pd.DataFrame(list(fi_gain.items()), columns=['Feature','Gain']).sort_values('Gain',ascending=False)

plt.figure(figsize=(8,5))
plt.barh(fi_df['Feature'][::-1], fi_df['Gain'][::-1], color='steelblue')
plt.xlabel('Feature Importance (Gain)')
plt.title('XGBoost Feature Importance (Gain) – S&P 500 Return Prediction')
plt.tight_layout()
plt.savefig('q6_feature_importance.png', bbox_inches='tight'); plt.show()
print(f'Top feature: {fi_df.iloc[0]["Feature"]}  (economically: rolling returns capture short-term momentum)')

### Q6 – Why Random k-Fold CV is Invalid for Financial Time Series

Random k-fold CV randomly shuffles temporally ordered observations into folds, violating the causal arrow of time in financial data. The label at time t (21-day forward return) literally depends on prices at t+21. When CV assigns a validation observation at t and training observations at t+k (k>0), the model sees the future during training. Additionally, adjacent observations share autocorrelation (momentum, mean-reversion), so treating them as i.i.d. inflates apparent generalisability. The result is validation AUC-ROC estimates 5–20% above the true out-of-sample performance — a gap that causes substantial real-money losses. The correct approach is **walk-forward (expanding-window) validation**: training always uses strictly past data, validation uses future data, respecting the temporal ordering a live system would face.


---
## Question 7: Empirical Bias–Variance Decomposition [Medium]

**Summary:** We empirically estimate Bias² and Variance for five models spanning the complexity spectrum (Decision Stump → Unpruned Tree → Logistic Regression → Random Forest → XGBoost) using B=10 bootstrap iterations on the Pima Indians Diabetes dataset.


In [ ]:
# ============================================================
# Q7 – Bias-Variance Decomposition (Pima Indians Diabetes)
# ============================================================
df_pima = pd.read_csv('diabetes.csv')
print('Shape:', df_pima.shape)

X_pima = df_pima.drop('Outcome', axis=1).values
y_pima = df_pima['Outcome'].values

# Fixed 70/30 split
X_pima_tr, X_pima_te, y_pima_tr, y_pima_te = train_test_split(
    X_pima, y_pima, test_size=0.30, random_state=SEED)
print(f'Train: {len(X_pima_tr)}, Test: {len(X_pima_te)}')

In [ ]:
def bias_variance_decomp(model_class, model_kwargs, X_train, y_train, X_test, y_test, B=10):
    n_test = len(X_test)
    preds_matrix = np.zeros((B, n_test))
    rng_bv = np.random.RandomState(SEED)
    for b in range(B):
        idx = rng_bv.choice(len(X_train), len(X_train), replace=True)
        m = model_class(**model_kwargs)
        m.fit(X_train[idx], y_train[idx])
        preds_matrix[b] = m.predict(X_test)
    y_bar = preds_matrix.mean(axis=0)
    bias2    = np.mean((y_bar - y_test) ** 2)
    variance = np.mean(np.mean((preds_matrix - y_bar) ** 2, axis=0))
    try:
        test_auc = roc_auc_score(y_test, y_bar)
    except:
        test_auc = float('nan')
    return bias2, variance, test_auc

models_bv = [
    ('Logistic Regression', LogisticRegression, {'solver':'lbfgs','max_iter':1000,'random_state':SEED}),
    ('Decision Stump',      DecisionTreeClassifier, {'max_depth':1,'random_state':SEED}),
    ('Unpruned Tree',       DecisionTreeClassifier, {'max_depth':None,'random_state':SEED}),
    ('Random Forest',       RandomForestClassifier, {'n_estimators':100,'random_state':SEED,'n_jobs':-1}),
    ('XGBoost',             xgb.XGBClassifier, {'random_state':SEED,'n_jobs':-1,'verbosity':0,'eval_metric':'logloss'}),
]

bv_results = []
for name, cls, kwargs in models_bv:
    print(f'  Bootstrap B=10 for: {name}...')
    b2, var, auc = bias_variance_decomp(cls, kwargs, X_pima_tr, y_pima_tr, X_pima_te, y_pima_te, B=10)
    bv_results.append({'Model':name, 'Bias²':round(b2,5), 'Variance':round(var,5),
                       'Bias²+Var':round(b2+var,5), 'Test AUC-ROC':round(auc,4)})

bv_df = pd.DataFrame(bv_results)
print('\n--- Q7: Bias²–Variance Summary Table ---')
print(bv_df.to_string(index=False))

In [ ]:
# Stacked bar chart
x = np.arange(len(bv_df))
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x, bv_df['Bias²'],    0.5, label='Bias²',    color='steelblue')
ax.bar(x, bv_df['Variance'], 0.5, bottom=bv_df['Bias²'], label='Variance', color='tomato')
ax.set_xticks(x)
ax.set_xticklabels(bv_df['Model'], rotation=15, ha='right')
ax.set_ylabel('Error Decomposition')
ax.set_title('Empirical Bias² and Variance – Pima Diabetes (B=10 Bootstraps)')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('q7_bias_variance.png', bbox_inches='tight'); plt.show()

### Q7 – Bagging vs. Boosting on Bias–Variance

**Bagging (Random Forest):** The formal variance of the average of B correlated predictions is **σ²ρ + σ²(1−ρ)/B**, where ρ is average pairwise tree correlation. As B→∞, the (1−ρ)σ²/B term vanishes, dramatically reducing variance. However, bagging does not change the mean prediction — bias equals that of a single tree. Since trees are low-bias, Random Forest achieves low bias AND low variance.

**Boosting (XGBoost):** Each successive tree fits the residual errors of the ensemble, explicitly reducing bias at each stage. Simultaneously, XGBoost's regularisation (L1, L2, subsampling) controls variance. The result is simultaneous bias AND variance reduction.


---
## Question 8: Gradient Boosting with Uncertainty Estimation for Return Prediction [Hard]

**Summary:** Capstone question integrating gradient boosting, feature engineering, expanding-window CV, and uncertainty estimation via ensemble disagreement on NIFTY-50. We use σᵢ (std across M=20 bootstrap models) as a model uncertainty proxy that directly informs uncertainty-aware position sizing.


In [ ]:
# ============================================================
# Q8 – NIFTY-50 (rohanrao/nifty50-stock-market-data)
# ============================================================
import glob

dest = r'D:\Avtansh\Semester 4\Summer Project Stamatics UAPOML\Ass3'
# Individual stock CSVs
nifty_files = [f for f in glob.glob(dest + r'\*.csv')
               if os.path.basename(f).isupper() or
               any(s in os.path.basename(f) for s in
               ['ADANI','ASIAN','AXIS','BAJAJ','BHART','BPCL','BRIT','CIPLA',
                'COAL','DRREDDY','EICHER','GAIL','GRASIM','HCL','HDFC','HERO',
                'HIND','ICICI','INDUS','INFY','IOC','ITC','JSW','KOTAK','LT',
                'MARUTI','MM','NESTLE','NTPC','ONGC','POWER','RELIANCE','SBIN',
                'SHREE','SUNPHARMA','TATA','TCS','TECHM','TITAN','ULTRA','UPL','VEDL','WIPRO','ZEEL'])]

print(f'Found {len(nifty_files)} NIFTY stock files')
dfs_nifty = []
for f in nifty_files:
    try:
        tmp = pd.read_csv(f, parse_dates=['Date'])
        tmp.columns = [c.strip().lower() for c in tmp.columns]
        if 'date' in tmp.columns and 'close' in tmp.columns:
            tmp['stock'] = os.path.basename(f).replace('.csv','')
            dfs_nifty.append(tmp)
    except Exception:
        pass

df_nifty = pd.concat(dfs_nifty, ignore_index=True)
print('Combined shape:', df_nifty.shape)

In [ ]:
# Feature Engineering
df_nifty = df_nifty.sort_values(['stock','date']).reset_index(drop=True)
df_nifty['daily_ret'] = df_nifty.groupby('stock')['close'].pct_change()

df_nifty['ret_5d']  = df_nifty.groupby('stock')['close'].pct_change(5)
df_nifty['ret_21d'] = df_nifty.groupby('stock')['close'].pct_change(21)
df_nifty['ret_63d'] = df_nifty.groupby('stock')['close'].pct_change(63)
df_nifty['vol_21d'] = df_nifty.groupby('stock')['daily_ret'].transform(
    lambda x: x.rolling(21).std())
df_nifty['vol_63d'] = df_nifty.groupby('stock')['daily_ret'].transform(
    lambda x: x.rolling(63).std())
df_nifty['p_52wk_hi'] = df_nifty.groupby('stock')['close'].transform(
    lambda x: x / x.rolling(252).max())
df_nifty['p_52wk_lo'] = df_nifty.groupby('stock')['close'].transform(
    lambda x: x / (x.rolling(252).min() + 1e-8))
df_nifty['vol_zscore'] = df_nifty.groupby('stock')['volume'].transform(
    lambda x: (x - x.rolling(21).mean()) / (x.rolling(21).std() + 1e-8))

df_nifty['fwd_ret'] = df_nifty.groupby('stock')['close'].transform(
    lambda x: x.pct_change(21).shift(-21))
df_nifty['target'] = (df_nifty['fwd_ret'] > 0).astype(int)

nifty_feats = ['ret_5d','ret_21d','ret_63d','vol_21d','vol_63d',
               'p_52wk_hi','p_52wk_lo','vol_zscore']
df_nifty.dropna(subset=nifty_feats+['target'], inplace=True)
print(f'Clean: {df_nifty.shape} | Date range: {df_nifty["date"].min().date()} → {df_nifty["date"].max().date()}')

In [ ]:
# Sort chronologically
df_nifty = df_nifty.sort_values('date').reset_index(drop=True)
X_nifty  = df_nifty[nifty_feats].values
y_nifty  = df_nifty['target'].values

# Time-based 80/20 split
split_pt = int(0.80 * len(X_nifty))
X_ntr, X_nte = X_nifty[:split_pt], X_nifty[split_pt:]
y_ntr, y_nte = y_nifty[:split_pt], y_nifty[split_pt:]
print(f'Train: {len(X_ntr)} | Test: {len(X_nte)}')

# Expanding-window CV (min 3 folds)
tscv = TimeSeriesSplit(n_splits=2)
hp_list = [
    {'n_estimators':100,'max_depth':2,'learning_rate':0.05},
    {'n_estimators':200,'max_depth':3,'learning_rate':0.01},
    {'n_estimators':500,'max_depth':3,'learning_rate':0.1},
    {'n_estimators':200,'max_depth':5,'learning_rate':0.05},
]

cv_res_nifty = []
for hp in hp_list:
    fold_aucs = []
    for tr_i, val_i in tscv.split(X_ntr):
        mc = GradientBoostingClassifier(
            n_estimators=hp['n_estimators'], max_depth=hp['max_depth'],
            learning_rate=hp['learning_rate'], random_state=SEED)
        mc.fit(X_ntr[tr_i], y_ntr[tr_i])
        try:
            fold_aucs.append(roc_auc_score(y_ntr[val_i], mc.predict_proba(X_ntr[val_i])[:,1]))
        except:
            fold_aucs.append(float('nan'))
    mean_a = round(np.nanmean(fold_aucs), 4)
    hp_row = dict(hp); hp_row['mean_val_AUC'] = mean_a
    cv_res_nifty.append(hp_row)
    print(f"  n_est={hp['n_estimators']}, depth={hp['max_depth']}, lr={hp['learning_rate']} → AUC={mean_a}")

cv_df_n = pd.DataFrame(cv_res_nifty).sort_values('mean_val_AUC', ascending=False)
best_hp = cv_df_n.iloc[0]
print('\n--- Expanding-Window CV Results ---')
print(cv_df_n.to_string(index=False))

In [ ]:
# Train full model
gb_full = GradientBoostingClassifier(
    n_estimators=int(best_hp['n_estimators']),
    max_depth=int(best_hp['max_depth']),
    learning_rate=best_hp['learning_rate'], random_state=SEED)
gb_full.fit(X_ntr, y_ntr)
auc_full = roc_auc_score(y_nte, gb_full.predict_proba(X_nte)[:,1])
print(f'Full model Test AUC-ROC: {auc_full:.4f}')

# Top-5 feature selection by permutation importance
perm_n = permutation_importance(gb_full, X_nte, y_nte, n_repeats=10,
                                 scoring='roc_auc', random_state=SEED)
top5_idx = np.argsort(perm_n.importances_mean)[-5:]
top5_names = [nifty_feats[i] for i in top5_idx]
print('Top 5 features:', top5_names)

gb_top5 = GradientBoostingClassifier(
    n_estimators=int(best_hp['n_estimators']),
    max_depth=int(best_hp['max_depth']),
    learning_rate=best_hp['learning_rate'], random_state=SEED)
gb_top5.fit(X_ntr[:, top5_idx], y_ntr)
auc_top5 = roc_auc_score(y_nte, gb_top5.predict_proba(X_nte[:, top5_idx])[:,1])
print(f'\nFull model AUC-ROC:     {auc_full:.4f}')
print(f'Top-5 feature AUC-ROC: {auc_top5:.4f}')

In [ ]:
# Uncertainty estimation via M=20 bootstrap models
M = 5
rng_unc = np.random.RandomState(SEED)
boot_probs = np.zeros((M, len(X_nte)))

for m in range(M):
    idx_b = rng_unc.choice(len(X_ntr), len(X_ntr), replace=True)
    gb_b = GradientBoostingClassifier(
        n_estimators=int(best_hp['n_estimators']),
        max_depth=int(best_hp['max_depth']),
        learning_rate=best_hp['learning_rate'], random_state=m)
    gb_b.fit(X_ntr[idx_b], y_ntr[idx_b])
    boot_probs[m] = gb_b.predict_proba(X_nte)[:,1]

p_mean = boot_probs.mean(axis=0)
p_std  = boot_probs.std(axis=0)
top10_unc = np.argsort(p_std)[-10:]

plt.figure(figsize=(9, 6))
plt.scatter(p_mean, p_std, alpha=0.3, s=8, color='steelblue')
plt.scatter(p_mean[top10_unc], p_std[top10_unc], color='red', s=60, zorder=5,
            label='Top 10 highest σ')
for i, idx in enumerate(top10_unc):
    plt.annotate(f'#{i+1}', (p_mean[idx], p_std[idx]), fontsize=8,
                 xytext=(5,5), textcoords='offset points')
plt.xlabel('Mean Predicted Probability (p*ᵢ)')
plt.ylabel('Std Dev σᵢ (Uncertainty)')
plt.title('Uncertainty Estimation: p*ᵢ vs σᵢ – NIFTY-50 (M=20 bootstrap models)')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('q8_uncertainty.png', bbox_inches='tight'); plt.show()

In [ ]:
# Long-short strategy vs benchmark
fwd_ret_te = df_nifty['fwd_ret'].values[split_pt:]
dates_te   = df_nifty['date'].values[split_pt:]

df_te_s = pd.DataFrame({'date': dates_te, 'p_mean': p_mean, 'fwd_ret': fwd_ret_te})
df_te_s.dropna(subset=['fwd_ret'], inplace=True)

def daily_ls(group):
    if len(group) < 5: return np.nan
    k = max(1, int(0.10 * len(group)))
    g = group.sort_values('p_mean')
    return g.tail(k)['fwd_ret'].mean() - g.head(k)['fwd_ret'].mean()

strat_ret = df_te_s.groupby('date').apply(daily_ls).dropna().sort_index()
bench_ret = df_te_s.groupby('date')['fwd_ret'].mean().sort_index().reindex(strat_ret.index)

cum_strat = (1 + strat_ret).cumprod() - 1
cum_bench = (1 + bench_ret).cumprod() - 1

plt.figure(figsize=(11, 5))
plt.plot(cum_strat.index, cum_strat.values*100, label='L/S Strategy', color='tomato', lw=2)
plt.plot(cum_bench.index, cum_bench.values*100, label='Buy-and-Hold Benchmark', color='steelblue', lw=2)
plt.xlabel('Date'); plt.ylabel('Cumulative Return (%)')
plt.title('Long-Short Strategy vs. NIFTY-50 Buy-and-Hold Benchmark (Test Period)')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('q8_cumulative_returns.png', bbox_inches='tight'); plt.show()

### Q8 – Reflection: Key Risks of Live Deployment

1. **Overfitting to non-stationary data:** Financial return distributions undergo regime changes. NIFTY-50 patterns from 2000–2017 may not generalise to post-2017 regimes, causing severe out-of-sample losses when the market environment shifts.
2. **Look-ahead bias:** Features must be computed solely from information available at decision time. Adjusted prices (incorporating future dividends), index reconstitution data, or any forward-looking input creates look-ahead bias that inflates backtests.
3. **Transaction costs:** Bid-ask spreads, market impact, brokerage fees, and short-selling costs (50–150+ bps/round-trip) are ignored here and would eliminate apparent alpha in a daily-rebalanced strategy.
4. **σᵢ and position sizing (UAPOML theme):** σᵢ from ensemble disagreement is a model uncertainty proxy. In an uncertainty-aware portfolio, position weight ∝ (p*ᵢ − 0.5)/σᵢ — discounting signals where model confidence is low. This implements the core UAPOML principle: uncertainty should shrink position sizes, not just inform them.


---
## Bonus: Stacked Generalisation for Titanic Survival Prediction [Hard]

**Summary:** We implement stacking with 5 diverse base learners using 5-fold out-of-fold predictions, then train a Logistic Regression meta-learner on the OOF meta-features. Stacking exploits complementary model strengths to potentially outperform any individual learner.


In [ ]:
# ============================================================
# BONUS – Stacking (Titanic, heptapod/titanic → tested.csv)
# ============================================================
df_titan = pd.read_csv('tested.csv')
print('Shape:', df_titan.shape)
print('Columns:', df_titan.columns.tolist())

# Feature Engineering
df_titan['Age'].fillna(df_titan['Age'].median(), inplace=True)
df_titan['Fare'].fillna(df_titan['Fare'].median(), inplace=True)
df_titan['Embarked'].fillna(df_titan['Embarked'].mode()[0], inplace=True)

df_titan['Sex_enc']    = (df_titan['Sex'] == 'male').astype(int)
df_titan['Embarked_S'] = (df_titan['Embarked'] == 'S').astype(int)
df_titan['Embarked_C'] = (df_titan['Embarked'] == 'C').astype(int)

titan_feats = ['Pclass','Sex_enc','Age','SibSp','Parch','Fare','Embarked_S','Embarked_C']
X_titan = df_titan[titan_feats].values
y_titan = df_titan['Survived'].values

# 80/20 split
X_ttr, X_tte, y_ttr, y_tte = train_test_split(
    X_titan, y_titan, test_size=0.20, random_state=SEED, stratify=y_titan)

scaler_stk = StandardScaler()
X_ttr_sc = scaler_stk.fit_transform(X_ttr)
X_tte_sc = scaler_stk.transform(X_tte)
print(f'Train: {len(X_ttr)}, Test: {len(X_tte)}')

In [ ]:
# Level-0 base learners + 5-fold OOF predictions
base_learners = {
    'Logistic Regression': LogisticRegression(solver='lbfgs', max_iter=1000, random_state=SEED),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=SEED),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
    'XGBoost':             xgb.XGBClassifier(n_estimators=100, random_state=SEED, verbosity=0,
                                              n_jobs=-1, eval_metric='logloss'),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
}

n_base  = len(base_learners)
n_ttr   = len(X_ttr_sc)
n_tte   = len(X_tte_sc)
skf5    = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_preds  = np.zeros((n_ttr, n_base))
test_preds = np.zeros((n_tte, n_base))
base_aucs, base_accs = {}, {}

for j, (name, model) in enumerate(base_learners.items()):
    fold_te = np.zeros((n_tte, 5))
    for fold, (tr_i, val_i) in enumerate(skf5.split(X_ttr_sc, y_ttr)):
        mc = type(model)(**model.get_params())
        mc.fit(X_ttr_sc[tr_i], y_ttr[tr_i])
        oof_preds[val_i, j] = mc.predict_proba(X_ttr_sc[val_i])[:,1]
        fold_te[:, fold]    = mc.predict_proba(X_tte_sc)[:,1]
    test_preds[:, j] = fold_te.mean(axis=1)

    # Individual model (full train)
    model.fit(X_ttr_sc, y_ttr)
    prob = model.predict_proba(X_tte_sc)[:,1]
    base_aucs[name] = round(roc_auc_score(y_tte, prob), 4)
    base_accs[name] = round(accuracy_score(y_tte, (prob>=0.5).astype(int)), 4)
    print(f'{name:22} | AUC-ROC: {base_aucs[name]:.4f} | Acc: {base_accs[name]:.4f}')

In [ ]:
# Level-1 meta-learner
meta_lr = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=SEED)
meta_lr.fit(oof_preds, y_ttr)

meta_prob = meta_lr.predict_proba(test_preds)[:,1]
meta_auc  = round(roc_auc_score(y_tte, meta_prob), 4)
meta_acc  = round(accuracy_score(y_tte, (meta_prob>=0.5).astype(int)), 4)
print(f'\nStacked Ensemble | AUC-ROC: {meta_auc:.4f} | Acc: {meta_acc:.4f}')

# Comparison table
rows = [{'Model':n,'AUC-ROC':base_aucs[n],'Accuracy':base_accs[n]} for n in base_learners]
rows.append({'Model':'Stacked Ensemble','AUC-ROC':meta_auc,'Accuracy':meta_acc})
print('\n--- Bonus: Base Learners vs. Stacked Ensemble ---')
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Meta-learner coefficient plot
coefs = meta_lr.coef_[0]
names = list(base_learners.keys())
plt.figure(figsize=(8,4))
plt.barh(names, coefs, color=['steelblue' if c>=0 else 'tomato' for c in coefs])
plt.xlabel('Meta-Learner Coefficient')
plt.title('Meta-Learner Coefficients – Stacking (Titanic)')
plt.grid(True, alpha=0.3, axis='x'); plt.tight_layout()
plt.savefig('bonus_meta_coef.png', bbox_inches='tight'); plt.show()

top_base = names[np.argmax(np.abs(coefs))]
print(f'Highest-weighted base learner: {top_base} (coef = {coefs[np.argmax(np.abs(coefs))]:.4f})')

### Bonus – Why OOF Predictions Must Be Used (Data Leakage in Stacking)

Out-of-fold predictions are essential because the meta-learner needs predictions the base learners did not see during training. If in-sample predictions were used: (1) overfit base learners (e.g., unpruned trees) produce near-perfect in-sample predictions by memorisation; (2) the meta-learner learns to heavily trust these overfit models; (3) in deployment, the overfit model predicts poorly on new data; (4) the stacked ensemble inherits this poor generalisation. OOF predictions provide an unbiased estimate of each base learner's true out-of-sample performance, allowing the meta-learner to assign weights proportional to genuine generalisability.


In [ ]:
# ============================================================
# COMPLETION SUMMARY
# ============================================================
import os
print('='*60)
print('ALL QUESTIONS COMPLETED SUCCESSFULLY')
print('='*60)
print('\nPlots saved:')
for f in sorted(os.listdir('.')):
    if f.endswith('.png'):
        size = os.path.getsize(f)
        print(f'  {f:40s}  ({size/1024:.1f} KB)')
print('\nSubmission files:')
for f in ['UAPOML_W3_Solution.ipynb', 'UAPOML_W3_Written_Report.pdf']:
    if os.path.exists(f):
        print(f'  ✓ {f}')
print('\nDone!')